# 02 - Flattening Layer (Phase 2)

Purpose: rebuild deterministic parquet contracts from read-only Mongo base collections.

Inputs: `players`, `bet_transactions`, `deposittransactions`, `withdrawaltransactions`, `bonustransactions`, `useractivitylogs`, `loginlogs`, plus identity/account support collections.

Outputs: parquet files and markdown reports under `data/`.

Core production logic lives in `src/frauddet/flatten.py`; this notebook only orchestrates, previews, and reconciles.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from frauddet import config
from frauddet.flatten import OUTPUT_FILES, rebuild_all_flattened_outputs

pd.set_option("display.max_columns", 80)

## Rebuild

In [ ]:
result = rebuild_all_flattened_outputs(config.DATA_DIR)
result.raw_counts, result.parquet_counts, result.report_paths

## Schema Previews

In [ ]:
frames = {
    name: pd.read_parquet(config.DATA_DIR / filename)
    for name, filename in OUTPUT_FILES.items()
}

for name, frame in frames.items():
    print(f"\n## {name}: {len(frame):,} rows")
    print(frame.dtypes.astype(str).to_string())
    display(frame.head(3))

## Row Counts

In [ ]:
pd.DataFrame(
    {
        "raw_mongo_rows": pd.Series(result.raw_counts),
        "parquet_rows": pd.Series(result.parquet_counts),
    }
)

## Player Join Rates

In [ ]:
join_rows = []
for name, frame in frames.items():
    if "player_key" not in frame.columns:
        continue
    nulls = int(frame["player_key"].isna().sum())
    join_rows.append(
        {
            "table": name,
            "rows": len(frame),
            "null_player_key": nulls,
            "join_rate": None if len(frame) == 0 else 1 - nulls / len(frame),
        }
    )

pd.DataFrame(join_rows).sort_values("table")

## Reconciliation

In [ ]:
print((config.DATA_DIR / "flatten_reconciliation.md").read_text(encoding="utf-8"))

Expected deltas explained above:

- Withdrawals collapse from lifecycle documents to one row per `transactionId`.
- Deposits are not deduplicated; all statuses are kept and interpreted through boolean flags.
- Unjoined source rows are kept with `player_key = NULL` and `unjoined_class`.
- Bonus currency is assumed from `config.yaml` because `bonustransactions` has no currency field.

There are zero unexplained drops in the current run.

## Reports

In [ ]:
for name, path in result.report_paths.items():
    print(f"\n# {name}: {path}\n")
    print(path.read_text(encoding="utf-8"))

## FINDINGS — Notebook 02: flattening layer — 2026-06-12
- Verified: Phase 2 rebuild produced `players=78`, `bets=379`, `money=197`, `bonus=62`, `activity=3,841`, `logins=855` parquet rows from current Mongo. Money rows reconcile as 155 deposits plus 42 collapsed withdrawal transaction groups from 77 lifecycle docs.
- Verified: Unjoined rows are retained with `player_key=NULL`: bets 16, money 3, bonus 7, activity 254, logins 438. `unjoined_report.md` includes counts by class and sample source IDs, and explicitly calls out known orphan-wallet IDs `0751231231` and `0754321012`.
- Verified: Withdrawal anomaly resolved. Status x `toAccountId` shows 15 wallet-pointing docs, all failed/declined (`failed=11`, `declined=4`); 28 docs are missing `fromAccountId`, split between initial request rows and failed/declined reversal rows. Collapse rule remains status-order based and keeps failed vs declined distinct.
- Surprises: Current live counts differ from earlier Phase 1 notes because the dev Mongo data has changed since those notebooks were reviewed.
- Gaps: Bonus transactions still have no currency field; `currency=UGX` is an assumed default from `config.yaml`. Login staff rows are retained with null `player_key` and `user_type` preserved; downstream feature code should filter `user_type == "PLAYER"` before device features.
- Decisions needed: Operations still needs to confirm whether `deposittransactions.status == "manual_reconciliation"` credits the player. Until then it remains excluded from `is_money_in`.
- Updated assumptions: Later phases should read only these parquet contracts, filter money using canonical flags, and avoid money-vs-bet currency ratios because money is UGX while bets are INR.